In [17]:
from qiskit import QuantumCircuit
from qiskit import transpile
from qiskit_aer import AerSimulator
import numpy as np
from qiskit_ibm_runtime.fake_provider import FakeManilaV2

In [13]:
def Simon_Oracle(s):
    n = len(s)
    oracle= QuantumCircuit(2*n)
    for i in range(n):
        oracle.cx(i, n+i)
    s= s[::-1]
    ind=-1
    for i in range(n):
        if s[i]=='1':
            ind= i
            break
    if ind<0:
        return oracle
    for i in range(n):
        if s[i]=='1':
            oracle.cx(ind, i+n)
    return oracle

In [14]:
def Simon(s):
    n = len(s)
    circuit= QuantumCircuit(2*n,n)
    q=0
    for i in range(n):
        circuit.h(i)
    circuit.barrier()
    oracle= Simon_Oracle(s)
    circuit.compose(oracle, inplace=True)
    circuit.barrier()
    # circuit.measure(n, q)
    for i in range(n):
        circuit.h(i)
    for i in range(n):
        circuit.measure(i, i)
    return circuit

In [19]:
# sim= AerSimulator()
fake_backend = FakeManilaV2()
noisy_simulator= AerSimulator.from_backend(fake_backend)
s= '10'
circuit= Simon(s)
transpiled_circuit= transpile(circuit, noisy_simulator)
job= fake_backend.run(transpiled_circuit, shots= 1024)
results= job.result()
print(circuit)
answers= results.get_counts().keys()
print(answers)
print(results.get_counts())

     ┌───┐ ░                 ░ ┌───┐┌─┐   
q_0: ┤ H ├─░───■─────────────░─┤ H ├┤M├───
     ├───┤ ░   │             ░ ├───┤└╥┘┌─┐
q_1: ┤ H ├─░───┼────■────■───░─┤ H ├─╫─┤M├
     └───┘ ░ ┌─┴─┐  │    │   ░ └───┘ ║ └╥┘
q_2: ──────░─┤ X ├──┼────┼───░───────╫──╫─
           ░ └───┘┌─┴─┐┌─┴─┐ ░       ║  ║ 
q_3: ──────░──────┤ X ├┤ X ├─░───────╫──╫─
           ░      └───┘└───┘ ░       ║  ║ 
c: 2/════════════════════════════════╩══╩═
                                     0  1 
dict_keys(['10', '11', '00', '01'])
{'10': 4, '11': 7, '00': 515, '01': 498}
